# Convert raw DUA files to parquet

This notebook turns the raw files published by the Dirección Nacional de Aduanas (`data/raw/`) into a single, clean table of customs declarations (DUAs) that is easy to query.

## What the raw files are

Every file is a zip archive holding three XML files. The official description is in `docs/FormatoDUADiariosPublicos.htm`.

| File inside the zip | What it contains | What to do with it |
|---|---|---|
| **Ing** (*ingresados*) | Declarations **entered** during that period, with every item. | Add them to the table. |
| **Mod** (*modificados*) | Declarations from **earlier periods** that were **changed** during this period, with their complete, corrected list of items. | Throw away every item we already have for that declaration and keep the new ones instead. |
| **Anu** (*anulados*) | Declarations that were **cancelled** during this period. Only the year and the declaration number are listed. | Remove every item of that declaration. |

A few things are worth knowing about the contents:

- **One row = one item of a declaration**, not one declaration. A declaration that ships five different products appears as five rows.
- **A declaration is identified by its year plus its public number** (`DDANO_PRESE` + `DDNUME_CORRE_PUBLICO`). The public number repeats every year, so the year is always needed. It is not the real customs operation number.
- A **cancelled** declaration (*DUA anulado*) was voided and no longer counts as a customs operation, so its goods and values must not be added to any total. Cancellations are rare (well under 1% of declarations) and often refer to declarations from an earlier month or year, which is why the files must be processed in date order.
- A **modification** replaces the whole declaration, not a single item. The spec warns that items are not published in any fixed order, so we cannot match "old item 3" with "new item 3". We simply swap the full set of items.
- Imports (`I`), exports (`E`) and transit (`T`) are all mixed in the same files. Use `DDTIPO_REGI` to tell them apart.

## Which files we use

There are two kinds of zip in `data/raw/`:

- **Monthly** (`DMyyyymm.zip` / `dmyyyymm.zip`): one per month, available for every month from 2016-01 onwards. The current month's file is refreshed every day.
- **Daily** (`ddyyyymmdd.zip` / `DDyyyymmdd.zip`): one per day, available from 2019-05 onwards.

**We only use the monthly files.** We checked that a monthly file contains exactly the same declarations and cancellations as all of that month's daily files combined. The daily files also report edits made within the same month, but the monthly *Ing* file already shows those declarations in their final, edited form. The daily files therefore add nothing and would only make the process slower.

## How the final table is built

The spec asks us to replay the files in date order, as if we were keeping a database up to date by hand: for each month, first *Ing* (add), then *Mod* (replace), then *Anu* (remove).

Instead of replaying ~130 months one by one, we get the same result in a single step. For every declaration, we look at all the times it shows up (in any Ing, Mod or Anu file) and keep only its **most recent appearance**:

- If the most recent appearance is in an **Ing** or **Mod** file, we keep the items from that file.
- If the most recent appearance is in an **Anu** file, the declaration was cancelled and does not appear in the final table.

"Most recent" means the latest month. Within the same month, the order is Ing → Mod → Anu, as the spec says.

We checked these facts on the 2020–2026 files, and the approach relies on them:

- Each *Ing* file only contains declarations dated in that same month.
- A declaration never appears in more than one of the three files for the same month.
- A cancelled declaration never shows up again afterwards.
- Every declaration has a single date, regime and importer/exporter, even across imports, exports and transit, so year + number is a safe identifier.

Before 2020 the *Mod* and *Anu* files are published empty, so for 2016–2019 we only have the declarations as entered.

## What comes out

1. `data/interim/<year>/<yyyymm>/{ing,mod,anu}_<yyyymm>.parquet`: each monthly file converted as it is, without applying any rule. These files are reused on later runs, so only new or updated months are converted again.
2. `data/processed/dua_<start>-<end>.parquet`: **the final table**, one row per item of every declaration that is still valid, in its latest version. Two extra columns say which file that version came from: `FUENTE_PERIODO` (month, `yyyymm`) and `FUENTE_TIPO` (`ing` or `mod`).
3. `data/processed/dua_anulados_<start>-<end>.parquet`: the list of cancellations (year, number and the month they were cancelled), kept separately so it can still be studied.

## Columns and cleaning

- We keep the 41 columns that actually hold data, using their original names from the spec. The older files (2016–2019) also include about 160 extra fields that are always empty; these are dropped.
- `DDCODI_REGI` (regime) and `DDTIPO_OPERA` (sub-regime) only exist from 2020. `DDUNID_TRANS` only exists until 2019. In other years they are empty (null).
- Numbers are padded with spaces in the XML. We strip the spaces and store amounts, quantities, weights and percentages as decimal numbers. The date becomes a real date, and the year becomes an integer.
- Codes (countries, customs offices, units, NCM, tax IDs, etc.) stay as **text** so that leading zeros are kept (e.g. country `076` = Brazil).
- The 2016–2018 files write the country of origin without leading zeros (`76` instead of `076`). We add them back so the codes match across years.


In [3]:
import logging
import re
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
import polars as pl
from tqdm.auto import tqdm
from uruguay_dua_comex import utils

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

data_raw = utils.data_raw_dir()
data_interim = utils.data_interim_dir()
data_processed = utils.data_processed_dir()

# Monthly archives are named DMyyyymm.zip (older years) or dmyyyymm.zip (newer
# years). Daily archives (ddyyyymmdd.zip) are ignored: see the explanation above.
MONTHLY_ZIP_PATTERN = re.compile(r"^dm(\d{6})\.zip$", re.IGNORECASE)

# The three XML files inside each zip, e.g. DMIng201601.xml / dmmod202306.xml.
MEMBER_PATTERN = re.compile(r"^d[dm](ing|mod|anu)\d+\.xml$", re.IGNORECASE)

# Order in which the spec says the files of the same month must be applied.
KIND_ORDER = {"ing": 0, "mod": 1, "anu": 2}

# A declaration is identified by its year plus its public number.
DUA_KEY = ["DDANO_PRESE", "DDNUME_CORRE_PUBLICO"]

# Columns kept from the Ing/Mod files, in the order the spec lists them, with
# the type each one is stored as. Codes stay as text to keep leading zeros.
ITEM_COLUMNS = {
    "DDANO_PRESE": pl.Int16,  # Año
    "DDNUME_CORRE_PUBLICO": pl.String,  # Número público del DUA
    "DDFECH_INGSI": pl.Date,  # Fecha del DUA
    "DDTIPO_REGI": pl.String,  # Tipo de régimen: I / E / T
    "DDPAIS_ORIGE": pl.String,  # País de origen
    "DDPART_NANDI": pl.String,  # Nomenclatura Común del Mercosur (NCM)
    "DDPUER_EMBAR": pl.String,  # País de procedencia (impo) / destino (expo)
    "DDCONV_INTER": pl.String,  # Convenio internacional
    "DDCODI_LIBER": pl.String,  # Exoneración
    "DDQUNICOM": pl.Float64,  # Cantidad de unidades comerciales
    "DDTUNICOM": pl.String,  # Tipo de unidades comerciales
    "DDUNID_FIQTY": pl.Float64,  # Cantidad de unidades físicas
    "DDUNID_FIDES": pl.String,  # Tipo de unidades físicas
    "DDVAD_INCR": pl.Float64,  # Valor en aduana
    "DDIMA_DOLAR": pl.Float64,  # IMADUNI U$S
    "DDLIMA_DOLAR": pl.Float64,  # IMADUNI U$S liberado
    "DDRMI_DOLAR": pl.Float64,  # Recargo mínimo U$S
    "DDLRMI_DOLAR": pl.Float64,  # Recargo mínimo U$S liberado
    "DDRAD_DOLAR": pl.Float64,  # Recargo adicional U$S
    "DDLRAD_DOLAR": pl.Float64,  # Recargo adicional U$S liberado
    "DDRMO_DOLAR": pl.Float64,  # Recargo móvil U$S
    "DDLRMO_DOLAR": pl.Float64,  # Recargo móvil U$S liberado
    "DDIVA_DOLAR": pl.Float64,  # IVA U$S
    "DDLIVA_DOLAR": pl.Float64,  # IVA U$S liberado
    "DDLIVAA_DOLA": pl.Float64,  # IVA anticipo U$S liberado
    "DDIVAA_DOLAR": pl.Float64,  # IVA anticipo U$S
    "DDPOR_IMADUN": pl.Float64,  # Porcentaje IMADUNI
    "DDPOR_RMI": pl.Float64,  # Porcentaje recargo mínimo
    "DDPOR_RAD": pl.Float64,  # Porcentaje recargo adicional
    "DDPOR_IVA": pl.Float64,  # Porcentaje IVA
    "DDPOR_IVAA": pl.Float64,  # Porcentaje IVA anticipo
    "DDVIA_TRANSP": pl.String,  # Vía de transporte
    "DDUNID_TRANS": pl.String,  # Not in the spec; only published until 2019
    "DDADUAINGEGR": pl.String,  # Aduana de ingreso/egreso
    "DDTIPO_DOCUM": pl.String,  # Tipo de documento del importador/exportador
    "DDLIBR_TRIBU": pl.String,  # Documento del importador/exportador
    "DDPESO_BRUTO": pl.Float64,  # Peso bruto
    "DDPESO_NETO": pl.Float64,  # Peso neto
    "DDDNOMBRE": pl.String,  # Importador/exportador
    "DDCODI_REGI": pl.String,  # Régimen; only published from 2020
    "DDTIPO_OPERA": pl.String,  # Subrégimen; only published from 2020
}

# Anu files only list which declarations were cancelled. The year tag has a
# different name there (DFANOPRE); it is renamed to match the Ing/Mod files.
ANU_COLUMNS = {"DFANOPRE": "DDANO_PRESE", "DDNUME_CORRE_PUBLICO": "DDNUME_CORRE_PUBLICO"}

c:\Users\ferna\Documents\uruguay_dua_comex\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: convert each monthly zip

Each monthly zip is opened, and its three XML files are read one row at a time, so even the largest files (~650 MB of XML) don't fill up memory. Each file is saved as its own parquet in `data/interim/`, cleaned and typed but otherwise unchanged. No Ing/Mod/Anu rule is applied yet.

A month that was already converted is skipped, unless its zip is newer than the converted files. That happens with the current month, whose file is refreshed daily.

In [4]:
def find_monthly_zips(raw_dir: Path) -> dict[int, Path]:
    """Find every monthly zip under raw_dir.

    Args:
        raw_dir: Directory containing data/raw/<year>/<DMyyyymm|dmyyyymm>.zip archives.

    Returns:
        Mapping of period (yyyymm as an int) to its zip path, sorted by period.
    """
    zips = {}
    for path in raw_dir.rglob("*.zip"):
        match = MONTHLY_ZIP_PATTERN.match(path.name)
        if match:
            zips[int(match.group(1))] = path
    return dict(sorted(zips.items()))


def read_xml_rows(zf: zipfile.ZipFile, member: str, columns: list[str]) -> pl.DataFrame:
    """Read the rows of one XML file inside a zip into a table of raw strings.

    Rows are read one at a time and discarded right after, so memory stays low
    even for the largest files. Tags missing from a row (newer files leave out
    empty fields) become null.

    Args:
        zf: Open zip archive.
        member: Name of the XML file inside the archive.
        columns: Tags to keep; every other tag is ignored.

    Returns:
        One row per XML row, every column as a string.
    """
    data = {column: [] for column in columns}
    with zf.open(member) as xml_file:
        for _, element in ET.iterparse(xml_file):
            # Ing/Mod rows are <ROW_DUASDIA>, Anu rows are <ROW>.
            if element.tag not in ("ROW_DUASDIA", "ROW"):
                continue
            values = {child.tag: child.text for child in element}
            for column in columns:
                data[column].append(values.get(column))
            element.clear()
    return pl.DataFrame(data, schema={column: pl.String for column in columns})


def blank_to_null(df: pl.DataFrame) -> pl.DataFrame:
    """Strip the padding spaces from every value and turn empty values into null."""
    return df.with_columns(pl.all().str.strip_chars().replace("", None))


def clean_items(df: pl.DataFrame) -> pl.DataFrame:
    """Cast an Ing/Mod table from raw strings to its final column types.

    Casts are strict, so an unexpected value (e.g. text in an amount column)
    stops the run instead of silently becoming null.
    """
    df = blank_to_null(df)
    casts = []
    for column, dtype in ITEM_COLUMNS.items():
        if dtype == pl.Date:
            casts.append(pl.col(column).str.to_date("%Y-%m-%d"))
        elif dtype != pl.String:
            casts.append(pl.col(column).cast(dtype, strict=True))
    # Older files drop the leading zeros of the country of origin ("76" instead
    # of "076"); put them back so the codes match across years.
    casts.append(pl.col("DDPAIS_ORIGE").str.zfill(3))
    return df.with_columns(casts)


def monthly_parquet_paths(period: int, interim_dir: Path) -> dict[str, Path]:
    """Where the three converted files of one month are stored."""
    month_dir = interim_dir / str(period)[:4] / str(period)
    return {kind: month_dir / f"{kind}_{period}.parquet" for kind in KIND_ORDER}


def convert_month_to_parquet(period: int, zip_path: Path, interim_dir: Path) -> dict[str, Path]:
    """Convert one monthly zip into three parquet files (ing, mod and anu).

    Args:
        period: Month of the zip, as yyyymm.
        zip_path: Path to the monthly zip archive.
        interim_dir: Root directory to write the monthly parquet files under.

    Returns:
        Mapping of file kind ("ing", "mod", "anu") to its parquet path.
    """
    output_paths = monthly_parquet_paths(period, interim_dir)

    # Skip months already converted, unless the zip was updated afterwards.
    zip_mtime = zip_path.stat().st_mtime
    if all(p.exists() and p.stat().st_mtime >= zip_mtime for p in output_paths.values()):
        return output_paths

    with zipfile.ZipFile(zip_path) as zf:
        members = {}
        for name in zf.namelist():
            match = MEMBER_PATTERN.match(Path(name).name)
            if match:
                members[match.group(1).lower()] = name
        missing = set(KIND_ORDER) - set(members)
        if missing:
            raise ValueError(f"{zip_path.name} is missing its {sorted(missing)} file(s).")

        # Every row records which file it came from; Step 2 uses this to know
        # which version of a declaration is the most recent.
        source_period = pl.lit(period, dtype=pl.Int32).alias("FUENTE_PERIODO")

        tables = {}
        for kind in ("ing", "mod"):
            items = clean_items(read_xml_rows(zf, members[kind], list(ITEM_COLUMNS)))
            tables[kind] = items.with_columns(source_period, pl.lit(kind).alias("FUENTE_TIPO"))

        cancelled = read_xml_rows(zf, members["anu"], list(ANU_COLUMNS)).rename(ANU_COLUMNS)
        cancelled = blank_to_null(cancelled).with_columns(
            pl.col("DDANO_PRESE").cast(pl.Int16, strict=True)
        )
        tables["anu"] = cancelled.with_columns(source_period, pl.lit("anu").alias("FUENTE_TIPO"))

    output_paths["ing"].parent.mkdir(parents=True, exist_ok=True)
    for kind, table in tables.items():
        table.write_parquet(output_paths[kind])
    return output_paths


def build_monthly_parquets(raw_dir: Path = data_raw, interim_dir: Path = data_interim) -> dict[int, dict[str, Path]]:
    """Convert every monthly zip under raw_dir into its three parquet files.

    Args:
        raw_dir: Directory containing the raw zip archives.
        interim_dir: Directory to write data/interim/<year>/<yyyymm>/{ing,mod,anu}_<yyyymm>.parquet.

    Returns:
        Mapping of period (yyyymm) to that month's parquet paths.
    """
    monthly_zips = find_monthly_zips(raw_dir)
    if not monthly_zips:
        logger.info("No monthly zip files found in the raw directory.")
        return {}

    periods = list(monthly_zips)
    logger.info(f"Converting {len(periods)} monthly zip files ({periods[0]} to {periods[-1]})...")

    converted = {}
    pbar = tqdm(monthly_zips.items(), desc="Converting months")
    for period, zip_path in pbar:
        pbar.set_postfix(current_file=zip_path.name)
        converted[period] = convert_month_to_parquet(period, zip_path, interim_dir)
    return converted

## Step 2: keep only the latest version of each declaration

All the monthly files are combined. For each declaration (year + number), we find its most recent appearance and keep only the items from that file. If the most recent appearance is a cancellation, there are no items to keep, so the declaration drops out on its own.

To compare appearances, each one gets a sort number made from its month and the file kind: `month × 10 + (0 for Ing, 1 for Mod, 2 for Anu)`. For example, a declaration entered in March 2023 and modified in May 2023 appears as `2023030` (Ing) and `2023051` (Mod). The larger number wins, so we keep the May version.

The work is done in streaming mode, so the full dataset never has to fit in memory.

In [ ]:
def event_order() -> pl.Expr:
    """Sort number of an appearance: month * 10 + position of its file kind (Ing, Mod, Anu)."""
    kind_position = pl.col("FUENTE_TIPO").replace_strict(KIND_ORDER, return_dtype=pl.Int64)
    return pl.col("FUENTE_PERIODO").cast(pl.Int64) * 10 + kind_position


def combine_monthly_parquets(
    converted: dict[int, dict[str, Path]], processed_dir: Path = data_processed
) -> None:
    """Build the final table of valid declarations and the list of cancellations.

    Args:
        converted: Mapping of period (yyyymm) to that month's parquet paths.
        processed_dir: Directory to write the processed parquet files.
    """
    if not converted:
        logger.info("No monthly parquet files found to combine.")
        return

    processed_dir.mkdir(parents=True, exist_ok=True)
    start_year, end_year = str(min(converted))[:4], str(max(converted))[:4]
    items_path = processed_dir / f"dua_{start_year}-{end_year}.parquet"
    cancelled_path = processed_dir / f"dua_anulados_{start_year}-{end_year}.parquet"

    items = pl.scan_parquet([paths[kind] for paths in converted.values() for kind in ("ing", "mod")])
    cancelled = pl.scan_parquet([paths["anu"] for paths in converted.values()])

    # For each declaration, the sort number of its most recent appearance in
    # any Ing, Mod or Anu file.
    latest_appearance = (
        pl.concat(
            [
                items.select(*DUA_KEY, event_order().alias("ORDEN")),
                cancelled.select(*DUA_KEY, event_order().alias("ORDEN")),
            ]
        )
        .group_by(DUA_KEY)
        .agg(pl.col("ORDEN").max().alias("ORDEN_ULTIMO"))
    )

    # Keep only the items that belong to that most recent appearance. When the
    # most recent appearance is a cancellation, no item matches it, so the
    # cancelled declaration disappears from the table.
    final_items = (
        items.with_columns(event_order().alias("ORDEN"))
        .join(latest_appearance, on=DUA_KEY, how="inner")
        .filter(pl.col("ORDEN") == pl.col("ORDEN_ULTIMO"))
        .drop("ORDEN", "ORDEN_ULTIMO")
    )

    logger.info(f"Writing {items_path}...")
    final_items.sink_parquet(items_path)

    logger.info(f"Writing {cancelled_path}...")
    (
        cancelled.select(*DUA_KEY, pl.col("FUENTE_PERIODO").alias("PERIODO_ANULACION"))
        .unique()
        .sort("PERIODO_ANULACION", *DUA_KEY)
        .sink_parquet(cancelled_path)
    )

    # Quick summary so each run can be sanity-checked at a glance.
    summary = pl.scan_parquet(items_path).select(
        pl.len().alias("items"),
        pl.struct(DUA_KEY).n_unique().alias("declarations"),
    ).collect()
    removed = cancelled.select(pl.struct(DUA_KEY).n_unique()).collect().item()
    logger.info(
        f"Done! {summary['items'][0]:,} items from {summary['declarations'][0]:,} valid "
        f"declarations; {removed:,} cancelled declarations listed separately."
    )

In [6]:
# Run the pipeline: convert each monthly zip (skipping months already converted),
# then combine everything into the final table.
converted = build_monthly_parquets()
combine_monthly_parquets(converted)

Converting 129 monthly zip files (201601 to 202609)...
Converting months: 100%|██████████| 129/129 [17:15<00:00,  8.03s/it, current_file=dm202609.zip]
Writing c:\Users\ferna\Documents\uruguay_dua_comex\data\processed\duas_2016-2026.parquet...
Writing c:\Users\ferna\Documents\uruguay_dua_comex\data\processed\duas_anulados_2016-2026.parquet...
Done! 20,473,963 items from 4,227,563 valid declarations; 11,896 cancelled declarations listed separately.
